# Analog Pressure Gauge Reader -- Training Notebook
**Pipeline**: YOLOv8n Detection -> Perspective Correction -> EasyOCR Scale Reading -> Needle Detection -> Pressure Reading

**Target deployment**: Raspberry Pi 4 (8GB) -- batch processing 400 images/hour

**Environment**: WSL Ubuntu, conda `vision-ml`, Python 3.10, PyTorch 2.6+cu124

## Phase 1: Setup & Data
### Cell 1: Environment & Imports

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
from pathlib import Path
from dataclasses import dataclass, field
from collections import namedtuple
import math
import json
import csv
import time
import re
import yaml
from PIL import Image
from ultralytics import YOLO
import easyocr
import albumentations as A
import pandas as pd
from tqdm import tqdm

# Verify environment
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"OpenCV: {cv2.__version__}")
print(f"NumPy: {np.__version__}")

# Initialize EasyOCR reader (GPU for training, CPU on Pi)
ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
print("EasyOCR initialized")

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

### Cell 2: Project Directory Structure

In [ ]:
# Project root -- adjust if needed
PROJECT_ROOT = Path.cwd()

# Create directory structure
dirs = {
    'dataset': PROJECT_ROOT / 'dataset',
    'models': PROJECT_ROOT / 'models',
    'exports': PROJECT_ROOT / 'exports',
    'results': PROJECT_ROOT / 'results',
    'rpi_deploy': PROJECT_ROOT / 'rpi_deploy',
}

for name, path in dirs.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {name:12s} -> {path}")

print("\nDirectory structure ready.")
print("\n>>> NEXT STEP: Download your Roboflow dataset (YOLOv8 format) into the 'dataset/' folder.")
print("    1. Go to universe.roboflow.com")
print("    2. Search 'pressure gauge' -> filter: Object Detection")
print("    3. Pick dataset with 500+ images, diverse gauge types")
print("    4. Download -> Format: YOLOv8 -> Extract into dataset/")

### Cell 3: Dataset Exploration

In [ ]:
# Load dataset config
data_yaml_path = dirs['dataset'] / 'data.yaml'
assert data_yaml_path.exists(), f"data.yaml not found at {data_yaml_path}. Download your Roboflow dataset first."

with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset Configuration:")
print(f"  Classes: {data_config.get('names', data_config.get('nc', 'unknown'))}")
print(f"  Number of classes: {data_config.get('nc', len(data_config.get('names', [])))}")

# Count images per split
for split in ['train', 'valid', 'test']:
    split_dir = dirs['dataset'] / split / 'images'
    if split_dir.exists():
        count = len(list(split_dir.glob('*')))
        print(f"  {split:6s}: {count} images")
    else:
        print(f"  {split:6s}: not found")

# Show sample images with annotations
train_images_dir = dirs['dataset'] / 'train' / 'images'
train_labels_dir = dirs['dataset'] / 'train' / 'labels'

sample_images = sorted(train_images_dir.glob('*'))[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Training Images with Annotations', fontsize=14)

for ax, img_path in zip(axes.flat, sample_images):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    ax.imshow(img)
    
    # Draw bounding boxes from label file
    label_path = train_labels_dir / (img_path.stem + '.txt')
    if label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls, cx, cy, bw, bh = int(parts[0]), *[float(x) for x in parts[1:5]]
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                           linewidth=2, edgecolor='lime', facecolor='none')
                    ax.add_patch(rect)
    
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Cell 4: Data Augmentation Analysis

In [ ]:
# Demonstrate augmentations that help with angle robustness
# YOLOv8 applies its own augmentations during training; this cell shows what they look like

augmentation_pipeline = A.Compose([
    A.Perspective(scale=(0.05, 0.15), p=1.0),
    A.Rotate(limit=20, p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.GaussNoise(p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
])

# Pick a sample image
sample_img_path = sorted(train_images_dir.glob('*'))[0]
sample_img = cv2.imread(str(sample_img_path))
sample_img = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Augmentation Examples (simulating different camera angles & conditions)', fontsize=14)

axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original', fontsize=10)
axes[0, 0].axis('off')

for ax in axes.flat[1:]:
    augmented = augmentation_pipeline(image=sample_img)['image']
    ax.imshow(augmented)
    ax.set_title('Augmented', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Note: YOLOv8 training applies built-in augmentation (mosaic, perspective, rotation).")
print("These settings are configured in the training cell below.")

### Cell 5: Dataset Validation

In [ ]:
# Validate dataset integrity
issues = []
total_images = 0
total_labels = 0
class_counts = {}

for split in ['train', 'valid', 'test']:
    img_dir = dirs['dataset'] / split / 'images'
    lbl_dir = dirs['dataset'] / split / 'labels'
    
    if not img_dir.exists():
        continue
    
    images = set(p.stem for p in img_dir.glob('*'))
    labels = set(p.stem for p in lbl_dir.glob('*.txt')) if lbl_dir.exists() else set()
    
    total_images += len(images)
    total_labels += len(labels)
    
    # Check for images without labels
    missing_labels = images - labels
    if missing_labels:
        issues.append(f"  {split}: {len(missing_labels)} images without labels")
    
    # Check for corrupt images
    for img_path in img_dir.glob('*'):
        try:
            img = cv2.imread(str(img_path))
            if img is None:
                issues.append(f"  {split}: corrupt image {img_path.name}")
        except Exception as e:
            issues.append(f"  {split}: error reading {img_path.name}: {e}")
    
    # Count class distribution
    for lbl_path in lbl_dir.glob('*.txt'):
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls = int(parts[0])
                    class_counts[cls] = class_counts.get(cls, 0) + 1

print(f"Total images: {total_images}")
print(f"Total labels: {total_labels}")
print(f"\nClass distribution:")
class_names = data_config.get('names', {})
for cls_id, count in sorted(class_counts.items()):
    name = class_names.get(cls_id, class_names[cls_id] if isinstance(class_names, list) and cls_id < len(class_names) else f"class_{cls_id}")
    print(f"  {name}: {count} annotations")

if issues:
    print(f"\n[!] Issues found ({len(issues)}):")
    for issue in issues:
        print(issue)
else:
    print("\n[OK] Dataset validation passed -- no issues found.")

# Plot class distribution
if class_counts:
    fig, ax = plt.subplots(figsize=(10, 4))
    names = [class_names.get(k, class_names[k] if isinstance(class_names, list) and k < len(class_names) else f"class_{k}") for k in sorted(class_counts.keys())]
    counts = [class_counts[k] for k in sorted(class_counts.keys())]
    ax.bar(names, counts, color='steelblue')
    ax.set_title('Annotation Class Distribution')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Phase 2: YOLOv8n Training
### Cell 6: Train YOLOv8n Gauge Detector

In [ ]:
# Train YOLOv8n for gauge face detection
model = YOLO('yolov8n.pt')  # Pretrained on COCO

results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    augment=True,
    # Angle-robustness augmentation
    degrees=15.0,         # +/-15 deg rotation
    perspective=0.001,    # perspective warp
    flipud=0.5,           # vertical flip probability
    fliplr=0.5,           # horizontal flip probability
    mosaic=1.0,           # mosaic augmentation
    mixup=0.1,            # mixup augmentation
    # Training config
    patience=20,          # early stopping patience
    save=True,
    project=str(dirs['models']),
    name='pressure_gauge_detector',
    exist_ok=True,
)

print("\nTraining complete!")
print(f"Best model: {dirs['models'] / 'pressure_gauge_detector' / 'weights' / 'best.pt'}")

### Cell 7: Training Results

In [ ]:
# Display training results
results_dir = dirs['models'] / 'pressure_gauge_detector'

# Show training curves
results_img = results_dir / 'results.png'
if results_img.exists():
    img = Image.open(results_img)
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Training Curves', fontsize=14)
    plt.tight_layout()
    plt.show()

# Show confusion matrix
confusion_img = results_dir / 'confusion_matrix.png'
if confusion_img.exists():
    img = Image.open(confusion_img)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Confusion Matrix', fontsize=14)
    plt.tight_layout()
    plt.show()

# Print CSV metrics summary
results_csv = results_dir / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print("Final epoch metrics:")
    print(df.tail(1).to_string(index=False))

### Cell 8: Model Evaluation

In [ ]:
# Load best model and evaluate on validation set
best_model_path = dirs['models'] / 'pressure_gauge_detector' / 'weights' / 'best.pt'
model = YOLO(str(best_model_path))

# Run validation
val_results = model.val(
    data=str(data_yaml_path),
    imgsz=640,
    device=0,
)

print(f"\nmAP@0.5:    {val_results.box.map50:.4f}")
print(f"mAP@0.5:0.95: {val_results.box.map:.4f}")
print(f"Precision:  {val_results.box.mp:.4f}")
print(f"Recall:     {val_results.box.mr:.4f}")

### Cell 9: Best Model Sanity Check

In [ ]:
# Quick inference on sample images to sanity check
test_images_dir = dirs['dataset'] / 'test' / 'images'
if not test_images_dir.exists():
    test_images_dir = dirs['dataset'] / 'valid' / 'images'

sample_test = sorted(test_images_dir.glob('*'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Model Predictions on Test Images', fontsize=14)

for ax, img_path in zip(axes.flat, sample_test):
    results = model(str(img_path), verbose=False)
    annotated = results[0].plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Print model size
model_size_mb = best_model_path.stat().st_size / (1024 * 1024)
print(f"\nModel size: {model_size_mb:.1f} MB")
print(f"Target for Pi: < 10 MB ({'OK' if model_size_mb < 10 else '[!] Consider pruning'})")

## Phase 3: Gauge Reading Pipeline -- OCR + Classical CV
### Cell 10: Perspective Correction

In [ ]:
def correct_perspective(gauge_crop):
    """Correct perspective distortion from off-angle photos.
    
    If the gauge face appears as an ellipse (viewed from an angle),
    this warps it back to a circle (head-on view).
    """
    gray = cv2.cvtColor(gauge_crop, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    
    # Dilate edges to connect nearby contours
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    edges = cv2.dilate(edges, kernel, iterations=1)
    
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return gauge_crop
    
    # Find the largest contour that could be the gauge face
    valid_contours = [c for c in contours if len(c) >= 5]  # fitEllipse needs >= 5 points
    if not valid_contours:
        return gauge_crop
    
    largest = max(valid_contours, key=cv2.contourArea)
    
    # Fit ellipse to the largest contour
    ellipse = cv2.fitEllipse(largest)
    center, (axis_a, axis_b), angle = ellipse
    
    # If nearly circular already (aspect ratio close to 1), skip correction
    aspect_ratio = min(axis_a, axis_b) / max(axis_a, axis_b) if max(axis_a, axis_b) > 0 else 1
    if aspect_ratio > 0.9:
        return gauge_crop
    
    h, w = gauge_crop.shape[:2]
    target_size = int(max(axis_a, axis_b))
    
    # Build affine transform to circularize the ellipse
    # Rotate so ellipse axes align with image axes, then scale minor axis
    M_rotate = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(gauge_crop, M_rotate, (w, h))
    
    # Scale to make it circular
    scale_x = max(axis_a, axis_b) / min(axis_a, axis_b) if axis_a < axis_b else 1.0
    scale_y = max(axis_a, axis_b) / min(axis_a, axis_b) if axis_b < axis_a else 1.0
    
    M_scale = np.float32([[scale_x, 0, center[0] * (1 - scale_x)],
                          [0, scale_y, center[1] * (1 - scale_y)]])
    corrected = cv2.warpAffine(rotated, M_scale, (int(w * scale_x), int(h * scale_y)))
    
    # Crop to gauge region
    cx, cy = int(center[0] * scale_x), int(center[1] * scale_y)
    radius = target_size // 2
    x1 = max(0, cx - radius)
    y1 = max(0, cy - radius)
    x2 = min(corrected.shape[1], cx + radius)
    y2 = min(corrected.shape[0], cy + radius)
    
    cropped = corrected[y1:y2, x1:x2]
    
    if cropped.size == 0:
        return gauge_crop
    
    return cropped

print("[OK] correct_perspective() defined")

### Cell 11: Gauge Center & Radius Detection

In [ ]:
def find_gauge_center(image):
    """Find the center and radius of the gauge face.
    
    Returns: (cx, cy, radius) or None if not found.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    
    h, w = gray.shape
    min_radius = min(h, w) // 6
    max_radius = min(h, w) // 2
    
    # Try Hough circles
    circles = cv2.HoughCircles(
        blurred,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=min(h, w) // 3,
        param1=100,
        param2=50,
        minRadius=min_radius,
        maxRadius=max_radius
    )
    
    if circles is not None:
        circles = np.round(circles[0]).astype(int)
        # Pick the circle closest to the image center
        img_center = np.array([w // 2, h // 2])
        dists = [np.linalg.norm(np.array([c[0], c[1]]) - img_center) for c in circles]
        best = circles[np.argmin(dists)]
        return int(best[0]), int(best[1]), int(best[2])
    
    # Fallback: contour-based
    edges = cv2.Canny(blurred, 30, 100)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        largest = max(contours, key=cv2.contourArea)
        (cx, cy), radius = cv2.minEnclosingCircle(largest)
        return int(cx), int(cy), int(radius)
    
    # Last resort: assume center of image
    return w // 2, h // 2, min(h, w) // 3

print("[OK] find_gauge_center() defined")

### Cell 12: OCR Scale Number Reading

In [ ]:
ScaleMarking = namedtuple('ScaleMarking', ['value', 'angle', 'position'])


def read_scale_numbers(image, center, radius, reader):
    """Read scale numbers from gauge face using OCR.
    
    Creates an annular mask where numbers typically appear (50-90% of radius),
    runs EasyOCR, filters for numeric results, and computes angular positions.
    
    Returns: list of ScaleMarking(value, angle, position)
    """
    cx, cy = center
    h, w = image.shape[:2]
    
    # Create annular mask for the number ring (50-90% of radius)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask, (cx, cy), int(radius * 0.92), 255, -1)
    cv2.circle(mask, (cx, cy), int(radius * 0.45), 0, -1)
    
    # Apply mask
    masked = cv2.bitwise_and(image, image, mask=mask)
    
    # Run EasyOCR
    results = reader.readtext(masked, allowlist='0123456789.')
    
    markings = []
    for bbox, text, conf in results:
        # Parse numeric value
        text = text.strip()
        try:
            value = float(text)
        except ValueError:
            continue
        
        if conf < 0.3:  # Skip low-confidence detections
            continue
        
        # Compute angular position of this number relative to gauge center
        # bbox is [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
        bbox_center_x = sum(pt[0] for pt in bbox) / 4
        bbox_center_y = sum(pt[1] for pt in bbox) / 4
        
        # Angle from center (0 deg = 12 o'clock, clockwise)
        dx = bbox_center_x - cx
        dy = -(bbox_center_y - cy)  # Flip Y (image Y goes down)
        angle = math.degrees(math.atan2(dx, dy)) % 360
        
        markings.append(ScaleMarking(
            value=value,
            angle=angle,
            position=(bbox_center_x, bbox_center_y)
        ))
    
    # Sort by angle
    markings.sort(key=lambda m: m.angle)
    
    return markings

print("[OK] read_scale_numbers() defined")

### Cell 13: Scale Interpolation Setup

In [ ]:
@dataclass
class GaugeScale:
    """Represents a gauge's scale derived from OCR-detected markings."""
    markings: list  # List of ScaleMarking
    min_value: float = 0.0
    max_value: float = 0.0
    start_angle: float = 0.0  # Angle at min_value
    end_angle: float = 0.0    # Angle at max_value
    sweep: float = 0.0        # Total angular sweep


def build_scale(markings):
    """Build a GaugeScale from OCR-detected markings.
    
    Determines scale range and angular mapping from detected numbers.
    Handles the common gauge layout where the scale sweeps ~270 deg with a
    dead zone at bottom-dead-center (6 o'clock position).
    """
    if len(markings) < 2:
        return None
    
    # Sort markings by value to determine scale direction
    by_value = sorted(markings, key=lambda m: m.value)
    
    min_val = by_value[0].value
    max_val = by_value[-1].value
    start_angle = by_value[0].angle
    end_angle = by_value[-1].angle
    
    # Compute sweep (handle wrap-around for gauges that cross 0/360 deg)
    # Most gauges sweep clockwise from min to max
    sweep = end_angle - start_angle
    if sweep < 0:
        sweep += 360  # Wrap around
    
    # Validate: values should increase monotonically with angle
    # (when following the sweep direction)
    angles_normalized = []
    for m in by_value:
        a = m.angle - start_angle
        if a < 0:
            a += 360
        angles_normalized.append(a)
    
    is_monotonic = all(a1 <= a2 for a1, a2 in zip(angles_normalized, angles_normalized[1:]))
    if not is_monotonic:
        print("  [!] Scale numbers are not monotonically ordered by angle -- results may be less accurate")
    
    return GaugeScale(
        markings=by_value,
        min_value=min_val,
        max_value=max_val,
        start_angle=start_angle,
        end_angle=end_angle,
        sweep=sweep,
    )

print("[OK] build_scale() defined")

### Cell 14: Needle Detection

In [ ]:
def detect_needle(image, center, radius):
    """Detect the needle angle on a gauge face.
    
    Uses color thresholding + line detection to find the needle,
    then computes its angle from the center.
    
    Returns: angle in degrees (0 deg = 12 o'clock, clockwise), or None.
    """
    cx, cy = center
    h, w = image.shape[:2]
    
    # Create mask: exclude center hub (15% radius) and outer ring (>90% radius)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask, (cx, cy), int(radius * 0.90), 255, -1)
    cv2.circle(mask, (cx, cy), int(radius * 0.15), 0, -1)
    
    # Convert to HSV and threshold for dark needle
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Method 1: Dark needle (black/dark gray)
    dark_mask = cv2.inRange(gray, 0, 60)
    
    # Method 2: Red needle
    red_lower1 = np.array([0, 100, 50])
    red_upper1 = np.array([10, 255, 255])
    red_lower2 = np.array([170, 100, 50])
    red_upper2 = np.array([180, 255, 255])
    red_mask = cv2.inRange(hsv, red_lower1, red_upper1) | cv2.inRange(hsv, red_lower2, red_upper2)
    
    # Combine needle masks
    needle_mask = cv2.bitwise_or(dark_mask, red_mask)
    needle_mask = cv2.bitwise_and(needle_mask, mask)
    
    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    needle_mask = cv2.morphologyEx(needle_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    needle_mask = cv2.morphologyEx(needle_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    
    # Detect lines using HoughLinesP
    lines = cv2.HoughLinesP(
        needle_mask,
        rho=1,
        theta=np.pi / 180,
        threshold=30,
        minLineLength=int(radius * 0.25),
        maxLineGap=int(radius * 0.1)
    )
    
    if lines is None:
        # Fallback: contour-based approach
        return _needle_from_contours(needle_mask, cx, cy, radius)
    
    # Filter lines: must pass near center, pick longest
    best_line = None
    best_score = 0
    
    for line in lines:
        x1, y1, x2, y2 = line[0]
        
        # Distance from center to line
        line_len = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        if line_len < 1:
            continue
        
        dist_to_center = abs((y2-y1)*cx - (x2-x1)*cy + x2*y1 - y2*x1) / line_len
        
        # Score: prefer long lines close to center
        if dist_to_center < radius * 0.3:
            score = line_len / (dist_to_center + 1)
            if score > best_score:
                best_score = score
                best_line = (x1, y1, x2, y2)
    
    if best_line is None:
        return _needle_from_contours(needle_mask, cx, cy, radius)
    
    x1, y1, x2, y2 = best_line
    
    # Determine which endpoint is the needle tip (farther from center)
    d1 = math.sqrt((x1 - cx)**2 + (y1 - cy)**2)
    d2 = math.sqrt((x2 - cx)**2 + (y2 - cy)**2)
    
    tip_x, tip_y = (x1, y1) if d1 > d2 else (x2, y2)
    
    # Calculate angle (0 deg = 12 o'clock, clockwise)
    dx = tip_x - cx
    dy = -(tip_y - cy)  # Flip Y
    angle = math.degrees(math.atan2(dx, dy)) % 360
    
    return angle


def _needle_from_contours(needle_mask, cx, cy, radius):
    """Fallback needle detection using contour analysis."""
    contours, _ = cv2.findContours(needle_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None
    
    # Find the longest thin contour
    best_contour = None
    best_length = 0
    
    for c in contours:
        arc_len = cv2.arcLength(c, closed=False)
        area = cv2.contourArea(c)
        
        # Needle-like: long perimeter, small area (thin)
        if arc_len > radius * 0.3 and (area < arc_len * radius * 0.2 or area > 0):
            if arc_len > best_length:
                best_length = arc_len
                best_contour = c
    
    if best_contour is None:
        return None
    
    # Find the point farthest from center
    dists = [math.sqrt((pt[0][0] - cx)**2 + (pt[0][1] - cy)**2) for pt in best_contour]
    tip_idx = np.argmax(dists)
    tip_x, tip_y = best_contour[tip_idx][0]
    
    dx = tip_x - cx
    dy = -(tip_y - cy)
    angle = math.degrees(math.atan2(dx, dy)) % 360
    
    return angle

print("[OK] detect_needle() defined")

### Cell 15: Angle-to-Reading Conversion

In [ ]:
def angle_to_reading(needle_angle, scale):
    """Convert needle angle to pressure reading using the OCR-derived scale.
    
    Uses piecewise linear interpolation between adjacent scale markings
    for maximum accuracy (handles non-linear scales).
    """
    if scale is None or len(scale.markings) < 2:
        return None
    
    # Normalize needle angle relative to scale start
    needle_rel = needle_angle - scale.start_angle
    if needle_rel < 0:
        needle_rel += 360
    
    # Normalize all marking angles relative to start
    mark_angles = []
    for m in scale.markings:
        a = m.angle - scale.start_angle
        if a < 0:
            a += 360
        mark_angles.append(a)
    
    # Clamp to scale range
    if needle_rel <= mark_angles[0]:
        return scale.markings[0].value
    if needle_rel >= mark_angles[-1]:
        return scale.markings[-1].value
    
    # Piecewise linear interpolation between adjacent markings
    for i in range(len(mark_angles) - 1):
        if mark_angles[i] <= needle_rel <= mark_angles[i + 1]:
            # Interpolate between marking i and i+1
            t = (needle_rel - mark_angles[i]) / (mark_angles[i + 1] - mark_angles[i])
            value = scale.markings[i].value + t * (scale.markings[i + 1].value - scale.markings[i].value)
            return round(value, 2)
    
    # Fallback: simple linear interpolation across full scale
    t = needle_rel / scale.sweep if scale.sweep > 0 else 0
    value = scale.min_value + t * (scale.max_value - scale.min_value)
    return round(value, 2)

print("[OK] angle_to_reading() defined")

### Cell 16: Full Pipeline Integration

In [ ]:
@dataclass
class GaugeReading:
    """Result from the gauge reading pipeline."""
    value: float = None
    scale_min: float = None
    scale_max: float = None
    needle_angle: float = None
    confidence: float = 0.0
    num_scale_markings: int = 0
    status: str = 'OK'
    annotated_image: np.ndarray = field(default=None, repr=False)


def read_gauge(image, yolo_model, ocr_reader, conf_threshold=0.5):
    """Full pipeline: detect gauge -> correct perspective -> OCR scale -> detect needle -> read value.
    
    Args:
        image: BGR image (numpy array)
        yolo_model: trained YOLO model
        ocr_reader: EasyOCR reader instance
        conf_threshold: minimum detection confidence
    
    Returns: GaugeReading
    """
    reading = GaugeReading()
    annotated = image.copy()
    
    # Step 1: Detect gauge face with YOLOv8
    results = yolo_model(image, verbose=False, conf=conf_threshold)
    boxes = results[0].boxes
    
    if len(boxes) == 0:
        reading.status = 'NO_GAUGE_DETECTED'
        reading.annotated_image = annotated
        return reading
    
    # Use highest confidence detection
    best_idx = boxes.conf.argmax()
    box = boxes.xyxy[best_idx].cpu().numpy().astype(int)
    det_conf = float(boxes.conf[best_idx])
    x1, y1, x2, y2 = box
    
    # Add padding (10%)
    pad_x = int((x2 - x1) * 0.1)
    pad_y = int((y2 - y1) * 0.1)
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(image.shape[1], x2 + pad_x)
    y2 = min(image.shape[0], y2 + pad_y)
    
    # Draw detection box
    cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
    
    # Step 2: Crop and correct perspective
    gauge_crop = image[y1:y2, x1:x2]
    corrected = correct_perspective(gauge_crop)
    
    # Step 3: Find gauge center
    center_result = find_gauge_center(corrected)
    if center_result is None:
        reading.status = 'CENTER_NOT_FOUND'
        reading.annotated_image = annotated
        return reading
    
    cx, cy, radius = center_result
    
    # Step 4: OCR scale numbers
    markings = read_scale_numbers(corrected, (cx, cy), radius, ocr_reader)
    reading.num_scale_markings = len(markings)
    
    if len(markings) < 2:
        reading.status = 'INSUFFICIENT_SCALE_MARKINGS'
        reading.annotated_image = annotated
        return reading
    
    # Step 5: Build scale model
    scale = build_scale(markings)
    if scale is None:
        reading.status = 'SCALE_BUILD_FAILED'
        reading.annotated_image = annotated
        return reading
    
    reading.scale_min = scale.min_value
    reading.scale_max = scale.max_value
    
    # Step 6: Detect needle
    needle_angle = detect_needle(corrected, (cx, cy), radius)
    if needle_angle is None:
        reading.status = 'NEEDLE_NOT_FOUND'
        reading.annotated_image = annotated
        return reading
    
    reading.needle_angle = needle_angle
    
    # Step 7: Convert to reading
    value = angle_to_reading(needle_angle, scale)
    reading.value = value
    reading.confidence = det_conf
    reading.status = 'OK'
    
    # Annotate image with reading
    label = f"{value:.1f} (conf: {det_conf:.2f})"
    cv2.putText(annotated, label, (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    
    # Draw needle line on annotation
    needle_tip_x = int(cx + radius * 0.8 * math.sin(math.radians(needle_angle)))
    needle_tip_y = int(cy - radius * 0.8 * math.cos(math.radians(needle_angle)))
    # Offset to original image coordinates
    cv2.circle(annotated, (x1 + cx, y1 + cy), 5, (255, 0, 0), -1)
    
    reading.annotated_image = annotated
    return reading

print("[OK] read_gauge() pipeline defined")

### Cell 17: Pipeline Testing & Visualization

In [ ]:
# Test the full pipeline on sample images
test_images_dir = dirs['dataset'] / 'test' / 'images'
if not test_images_dir.exists():
    test_images_dir = dirs['dataset'] / 'valid' / 'images'

test_images = sorted(test_images_dir.glob('*'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Full Pipeline Results', fontsize=14)

for ax, img_path in zip(axes.flat, test_images):
    img = cv2.imread(str(img_path))
    result = read_gauge(img, model, ocr_reader)
    
    display = cv2.cvtColor(result.annotated_image, cv2.COLOR_BGR2RGB)
    ax.imshow(display)
    
    if result.status == 'OK':
        title = f"Reading: {result.value:.1f}\nScale: {result.scale_min}-{result.scale_max}\nAngle: {result.needle_angle:.1f} deg"
    else:
        title = f"Status: {result.status}"
    
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Print detailed results
print("\nDetailed Results:")
print(f"{'Image':<30} {'Value':>8} {'Scale':>12} {'Angle':>8} {'Markings':>9} {'Status':>15}")
print("-" * 90)
for img_path in test_images:
    img = cv2.imread(str(img_path))
    result = read_gauge(img, model, ocr_reader)
    val_str = f"{result.value:.1f}" if result.value is not None else "N/A"
    scale_str = f"{result.scale_min}-{result.scale_max}" if result.scale_min is not None else "N/A"
    angle_str = f"{result.needle_angle:.1f} deg" if result.needle_angle is not None else "N/A"
    print(f"{img_path.name:<30} {val_str:>8} {scale_str:>12} {angle_str:>8} {result.num_scale_markings:>9} {result.status:>15}")

### Cell 18: Batch Processing Function

In [ ]:
def batch_process(image_folder, yolo_model, ocr_reader, output_csv, save_annotated=False, results_dir=None):
    """Process all images in a folder and output results to CSV.
    
    Args:
        image_folder: Path to folder containing gauge images
        yolo_model: trained YOLO model
        ocr_reader: EasyOCR reader instance
        output_csv: Path for output CSV file
        save_annotated: if True, save annotated images to results_dir
        results_dir: directory for annotated images (required if save_annotated=True)
    
    Returns: pandas DataFrame with results
    """
    image_folder = Path(image_folder)
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
    image_paths = sorted([p for p in image_folder.iterdir() 
                         if p.suffix.lower() in image_extensions])
    
    if not image_paths:
        print(f"No images found in {image_folder}")
        return pd.DataFrame()
    
    if save_annotated and results_dir:
        Path(results_dir).mkdir(parents=True, exist_ok=True)
    
    records = []
    times = []
    
    print(f"Processing {len(image_paths)} images...")
    
    for img_path in tqdm(image_paths, desc="Reading gauges"):
        t_start = time.time()
        
        img = cv2.imread(str(img_path))
        if img is None:
            records.append({
                'filename': img_path.name,
                'reading': None,
                'scale_min': None,
                'scale_max': None,
                'needle_angle': None,
                'confidence': 0.0,
                'num_markings': 0,
                'status': 'IMAGE_READ_ERROR',
                'time_sec': 0.0,
            })
            continue
        
        result = read_gauge(img, yolo_model, ocr_reader)
        elapsed = time.time() - t_start
        times.append(elapsed)
        
        records.append({
            'filename': img_path.name,
            'reading': result.value,
            'scale_min': result.scale_min,
            'scale_max': result.scale_max,
            'needle_angle': result.needle_angle,
            'confidence': result.confidence,
            'num_markings': result.num_scale_markings,
            'status': result.status,
            'time_sec': round(elapsed, 2),
        })
        
        # Save annotated image
        if save_annotated and results_dir and result.annotated_image is not None:
            out_path = Path(results_dir) / f"annotated_{img_path.name}"
            cv2.imwrite(str(out_path), result.annotated_image)
    
    # Create DataFrame and save CSV
    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    
    # Print summary
    ok_count = len(df[df['status'] == 'OK'])
    total = len(df)
    avg_time = np.mean(times) if times else 0
    
    print(f"\n{'='*50}")
    print(f"BATCH PROCESSING SUMMARY")
    print(f"{'='*50}")
    print(f"Total images:    {total}")
    print(f"Successful:      {ok_count} ({ok_count/total*100:.1f}%)")
    print(f"Failed:          {total - ok_count}")
    print(f"Avg time/image:  {avg_time:.2f} sec")
    print(f"Total time:      {sum(times):.1f} sec")
    if ok_count > 0:
        ok_df = df[df['status'] == 'OK']
        print(f"Reading range:   {ok_df['reading'].min():.1f} - {ok_df['reading'].max():.1f}")
    print(f"Results saved:   {output_csv}")
    
    # Status breakdown
    if total - ok_count > 0:
        print(f"\nFailure breakdown:")
        for status, count in df[df['status'] != 'OK']['status'].value_counts().items():
            print(f"  {status}: {count}")
    
    return df


# --- Run batch processing on test images ---
test_dir = dirs['dataset'] / 'test' / 'images'
if not test_dir.exists():
    test_dir = dirs['dataset'] / 'valid' / 'images'

df_results = batch_process(
    image_folder=test_dir,
    yolo_model=model,
    ocr_reader=ocr_reader,
    output_csv=dirs['results'] / 'test_results.csv',
    save_annotated=True,
    results_dir=dirs['results'] / 'annotated',
)

print("\nSample results:")
df_results.head(10)

## Phase 4: Export & Raspberry Pi Deployment
### Cell 19: ONNX Export

In [ ]:
# Export YOLOv8n to ONNX for Raspberry Pi deployment
best_model_path = dirs['models'] / 'pressure_gauge_detector' / 'weights' / 'best.pt'
export_model = YOLO(str(best_model_path))

# ONNX export (preferred for Pi -- uses OpenCV DNN backend)
onnx_path = export_model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    opset=12,
)

onnx_path = Path(onnx_path)
onnx_size_mb = onnx_path.stat().st_size / (1024 * 1024)
print(f"\nONNX model exported: {onnx_path}")
print(f"Model size: {onnx_size_mb:.1f} MB")
print(f"Pi-friendly: {'YES' if onnx_size_mb < 15 else '[!] Large for Pi'}")

# Copy to exports directory
import shutil
export_dest = dirs['exports'] / 'gauge_detector.onnx'
shutil.copy2(onnx_path, export_dest)
print(f"Copied to: {export_dest}")

### Cell 20: ONNX Verification

In [ ]:
# Verify ONNX model produces same results as PyTorch model
# Load ONNX via ultralytics (which uses OpenCV DNN or onnxruntime internally)
onnx_model = YOLO(str(export_dest))

# Compare on test images
test_images = sorted(test_dir.glob('*'))[:5]

print("Comparing PyTorch vs ONNX predictions:")
print(f"{'Image':<30} {'PyTorch boxes':>14} {'ONNX boxes':>11} {'Match':>6}")
print("-" * 65)

all_match = True
for img_path in test_images:
    # PyTorch
    pt_results = model(str(img_path), verbose=False)
    pt_boxes = len(pt_results[0].boxes)
    
    # ONNX
    onnx_results = onnx_model(str(img_path), verbose=False)
    onnx_boxes = len(onnx_results[0].boxes)
    
    match = pt_boxes == onnx_boxes
    if not match:
        all_match = False
    
    print(f"{img_path.name:<30} {pt_boxes:>14} {onnx_boxes:>11} {'OK' if match else 'DIFF':>6}")

print(f"\nOverall: {'All match' if all_match else '[!] Some mismatches (minor differences are normal)'}")

# CPU benchmark (simulates Pi performance)
print("\nCPU Inference Benchmark (simulating Pi performance):")
times = []
for img_path in test_images:
    t0 = time.time()
    _ = onnx_model(str(img_path), verbose=False, device='cpu')
    times.append(time.time() - t0)

print(f"  Average: {np.mean(times):.3f} sec/image")
print(f"  Note: Raspberry Pi 4 will be ~3-5x slower than this WSL CPU benchmark")
estimated_pi_time = np.mean(times) * 4
print(f"  Estimated Pi time: ~{estimated_pi_time:.1f} sec/image (YOLO detection only)")

### Cell 21: Generate RPi Deployment Package

In [ ]:
# Copy ONNX model to rpi_deploy
rpi_model_dest = dirs['rpi_deploy'] / 'gauge_detector.onnx'
shutil.copy2(export_dest, rpi_model_dest)
print(f"[OK] Model copied to {rpi_model_dest}")

print(f"\n[OK] Deployment package ready at:")
print(f"  {dirs['rpi_deploy']}")
print("\nContents:")
for f in sorted(dirs['rpi_deploy'].iterdir()):
    size = f.stat().st_size / 1024
    unit = 'KB'
    if size > 1024:
        size /= 1024
        unit = 'MB'
    print(f"  {f.name:<30} {size:.1f} {unit}")

print("\n>>> Transfer the entire rpi_deploy/ folder to your Raspberry Pi 4.")
print("    Then follow the instructions in README_PI.md.")

### Cell 22: Deployment Summary & Next Steps

In [ ]:
print("""
==================================================================
        PRESSURE GAUGE READER -- DEPLOYMENT SUMMARY
==================================================================

  Pipeline: YOLOv8n -> Perspective -> EasyOCR -> Needle -> Reading
  Target:   Raspberry Pi 4 (8GB), batch 400 images/hour

  Files to deploy (rpi_deploy/):
    * gauge_reader.py      -- Main inference script
    * gauge_detector.onnx  -- YOLOv8n model
    * requirements_pi.txt  -- Python dependencies
    * README_PI.md         -- Setup & usage guide

  RPi Usage:
    python3 gauge_reader.py --input ./photos/ --output results.csv
    python3 gauge_reader.py --single photo.jpg

==================================================================
""")